In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
df = pd.read_csv('data.csv')
print(df.head())

   Customer ID  Gender  Age           City Membership Type  Total Spend  \
0          101  Female   29       New York            Gold      1120.20   
1          102    Male   34    Los Angeles          Silver       780.50   
2          103  Female   43        Chicago          Bronze       510.75   
3          104    Male   30  San Francisco            Gold      1480.30   
4          105    Male   27          Miami          Silver       720.40   

   Items Purchased  Average Rating  Discount Applied  \
0               14             4.6              True   
1               11             4.1             False   
2                9             3.4              True   
3               19             4.7             False   
4               13             4.0              True   

   Days Since Last Purchase Satisfaction Level  
0                        25          Satisfied  
1                        18            Neutral  
2                        42        Unsatisfied  
3               

In [4]:
df.info()
df.describe()
df.isnull().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 350 entries, 0 to 349
Data columns (total 11 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Customer ID               350 non-null    int64  
 1   Gender                    350 non-null    object 
 2   Age                       350 non-null    int64  
 3   City                      350 non-null    object 
 4   Membership Type           350 non-null    object 
 5   Total Spend               350 non-null    float64
 6   Items Purchased           350 non-null    int64  
 7   Average Rating            350 non-null    float64
 8   Discount Applied          350 non-null    bool   
 9   Days Since Last Purchase  350 non-null    int64  
 10  Satisfaction Level        348 non-null    object 
dtypes: bool(1), float64(2), int64(4), object(4)
memory usage: 27.8+ KB


Customer ID                 0
Gender                      0
Age                         0
City                        0
Membership Type             0
Total Spend                 0
Items Purchased             0
Average Rating              0
Discount Applied            0
Days Since Last Purchase    0
Satisfaction Level          2
dtype: int64

### 3 CORE ANALYSES

##### - Customer Segmentation

        Group customers based on:

            - spending
            - frequency (Items Purchased)
            - recency (Days Since Last Purchase)


##### - Revenue Drivers


        Analyze:

            - membership type vs spend
            - discount vs spend
            - satisfaction vs spend
##### - Retention / Churn Proxy (refined)

        No have explicit churn, so to define it:
        If Days Since Last Purchase > 60 → At Risk Customer

        So now we can:

            - classify churn risk
            - analyze patterns

In [15]:
print(df['Total Spend'].describe())

count     350.000000
mean      845.381714
std       362.058695
min       410.800000
25%       502.000000
50%       775.200000
75%      1160.600000
max      1520.100000
Name: Total Spend, dtype: float64


##### Data Cleaning

###### Handle Missing

In [16]:
df['Satisfaction Level'].fillna(df['Satisfaction Level'].mode()[0], inplace=True)

/var/folders/nm/56h602q57rbg4b0lfs4nlphc0000gn/T/ipykernel_90718/3607365204.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['Satisfaction Level'].fillna(df['Satisfaction Level'].mode()[0], inplace=True)


###### Rename Columns

In [17]:
df.columns = df.columns.str.lower().str.replace(" ", "_")

###### Sanity Checks

In [18]:
df.duplicated().sum()

np.int64(0)

#### Feature Engineering

##### Customer Value Segment

In [19]:
df['customer_segment'] = pd.qcut(df['total_spend'], q=3, labels=['Low', 'Medium', 'High'])

##### Churn Risk

In [26]:
df['churn_risk'] = df.apply(
    lambda x: 'High' if (x['days_since_last_purchase'] > 60 or 
                         (x['days_since_last_purchase'] > 30 and x['satisfaction_level'] == 'Unsatisfied'))
    else ('Medium' if x['days_since_last_purchase'] > 30 else 'Low'),
    axis=1
)

##### Purchase Intensity

In [21]:
df['avg_spend_per_item'] = df['total_spend'] / df['items_purchased']

### Membership Type vs Spend

In [22]:
df.groupby('membership_type')['total_spend'].mean()

membership_type
Bronze     473.388793
Gold      1311.144444
Silver     748.432479
Name: total_spend, dtype: float64

In [25]:
df.groupby('membership_type')['avg_spend_per_item'].mean()

membership_type
Bronze    56.209825
Gold      74.775524
Silver    64.624438
Name: avg_spend_per_item, dtype: float64

### Churn Risk Distribution

In [27]:
df['churn_risk'].value_counts()

churn_risk
Low       226
High      116
Medium      8
Name: count, dtype: int64

In [28]:
df.groupby('churn_risk')['total_spend'].mean()

churn_risk
High      595.136207
Low       969.303540
Medium    973.150000
Name: total_spend, dtype: float64

### Segment vs Satisfaction

In [24]:
pd.crosstab(df['customer_segment'], df['satisfaction_level'])

satisfaction_level,Neutral,Satisfied,Unsatisfied
customer_segment,,,
Low,56,2,66
Medium,51,8,50
High,0,117,0
